# Phase 2 — Baseline Player Value Model

This notebook tests the first version of the NBA fantasy basketball 9-category player value model.

The model currently:

- Scores counting categories with z-scores
- Handles turnovers as a negative category
- Uses volume-weighted FG% impact
- Uses volume-weighted FT% impact
- Supports simple punt-category rankings

This is not yet a final fantasy ranking model. It is a baseline scoring engine for testing logic before adding real projections.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0]

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

from nba_fantasy.scoring import add_9cat_scores

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

PROJECT_ROOT

WindowsPath('C:/Users/graha/Documents/Data_Projects/nba-fantasy-decision-system')

In [2]:
players = pd.DataFrame(
    [
        {
            "player": "Nikola Jokic",
            "pts": 26.4,
            "reb": 12.4,
            "ast": 9.0,
            "stl": 1.4,
            "blk": 0.9,
            "threes": 1.1,
            "fg_pct": 0.583,
            "fga": 17.9,
            "ft_pct": 0.817,
            "fta": 5.5,
            "to": 3.0,
        },
        {
            "player": "Shai Gilgeous-Alexander",
            "pts": 30.1,
            "reb": 5.5,
            "ast": 6.2,
            "stl": 2.0,
            "blk": 0.9,
            "threes": 1.3,
            "fg_pct": 0.535,
            "fga": 20.0,
            "ft_pct": 0.875,
            "fta": 8.7,
            "to": 2.2,
        },
        {
            "player": "Anthony Davis",
            "pts": 24.7,
            "reb": 12.6,
            "ast": 3.5,
            "stl": 1.2,
            "blk": 2.3,
            "threes": 0.4,
            "fg_pct": 0.556,
            "fga": 17.2,
            "ft_pct": 0.816,
            "fta": 6.8,
            "to": 2.1,
        },
        {
            "player": "Stephen Curry",
            "pts": 26.4,
            "reb": 4.5,
            "ast": 5.1,
            "stl": 0.7,
            "blk": 0.4,
            "threes": 4.8,
            "fg_pct": 0.450,
            "fga": 19.5,
            "ft_pct": 0.923,
            "fta": 4.4,
            "to": 2.8,
        },
        {
            "player": "Myles Turner",
            "pts": 17.1,
            "reb": 6.9,
            "ast": 1.3,
            "stl": 0.5,
            "blk": 1.9,
            "threes": 1.5,
            "fg_pct": 0.524,
            "fga": 11.8,
            "ft_pct": 0.773,
            "fta": 3.6,
            "to": 1.4,
        },
        {
            "player": "DeMar DeRozan",
            "pts": 22.1,
            "reb": 4.3,
            "ast": 5.3,
            "stl": 1.1,
            "blk": 0.6,
            "threes": 0.9,
            "fg_pct": 0.480,
            "fga": 17.3,
            "ft_pct": 0.853,
            "fta": 6.5,
            "to": 1.7,
        },
        {
            "player": "Nic Claxton",
            "pts": 11.8,
            "reb": 9.9,
            "ast": 2.1,
            "stl": 0.7,
            "blk": 2.1,
            "threes": 0.0,
            "fg_pct": 0.629,
            "fga": 7.7,
            "ft_pct": 0.551,
            "fta": 2.9,
            "to": 1.3,
        },
        {
            "player": "Klay Thompson",
            "pts": 17.9,
            "reb": 3.3,
            "ast": 2.3,
            "stl": 0.6,
            "blk": 0.5,
            "threes": 3.5,
            "fg_pct": 0.432,
            "fga": 15.0,
            "ft_pct": 0.927,
            "fta": 1.0,
            "to": 1.5,
        },
    ]
)

players

,player,pts,reb,ast,stl,blk,threes,fg_pct,fga,ft_pct,fta,to
0,Nikola Jokic,26.4,12.4,9.0,1.4,0.9,1.1,0.583,17.9,0.817,5.5,3.0
1,Shai Gilgeous-Alexander,30.1,5.5,6.2,2.0,0.9,1.3,0.535,20.0,0.875,8.7,2.2
2,Anthony Davis,24.7,12.6,3.5,1.2,2.3,0.4,0.556,17.2,0.816,6.8,2.1
3,Stephen Curry,26.4,4.5,5.1,0.7,0.4,4.8,0.450,19.5,0.923,4.4,2.8
4,Myles Turner,17.1,6.9,1.3,0.5,1.9,1.5,0.524,11.8,0.773,3.6,1.4
5,DeMar DeRozan,22.1,4.3,5.3,1.1,0.6,0.9,0.480,17.3,0.853,6.5,1.7
6,Nic Claxton,11.8,9.9,2.1,0.7,2.1,0.0,0.629,7.7,0.551,2.9,1.3
7,Klay Thompson,17.9,3.3,2.3,0.6,0.5,3.5,0.432,15.0,0.927,1.0,1.5


In [3]:
balanced = add_9cat_scores(players)

balanced[
    [
        "player",
        "total_9cat_z",
        "pts_z",
        "reb_z",
        "ast_z",
        "stl_z",
        "blk_z",
        "threes_z",
        "fg_impact_z",
        "ft_impact_z",
        "to_z",
    ]
].round(2)

,player,total_9cat_z,pts_z,reb_z,ast_z,stl_z,blk_z,threes_z,fg_impact_z,ft_impact_z,to_z
0,Shai Gilgeous-Alexander,4.28,1.42,-0.55,0.77,2.06,-0.42,-0.25,0.38,1.21,-0.33
1,Nikola Jokic,3.64,0.77,1.43,1.94,0.79,-0.42,-0.39,1.30,-0.12,-1.67
2,Anthony Davis,3.07,0.47,1.49,-0.36,0.37,1.52,-0.84,0.74,-0.14,-0.17
3,Stephen Curry,-1.20,0.77,-0.84,0.31,-0.69,-1.11,2.04,-1.46,1.11,-1.33
4,DeMar DeRozan,-1.40,0.01,-0.90,0.40,0.16,-0.83,-0.52,-0.71,0.49,0.50
5,Myles Turner,-1.98,-0.88,-0.15,-1.27,-1.11,0.97,-0.12,0.13,-0.54,1.00
6,Nic Claxton,-2.57,-1.81,0.71,-0.94,-0.69,1.25,-1.11,1.02,-2.17,1.17
7,Klay Thompson,-3.85,-0.74,-1.18,-0.86,-0.90,-0.97,1.19,-1.39,0.16,0.83


In [4]:
punt_ft = add_9cat_scores(players, punt_categories=["ft_impact"])
punt_to = add_9cat_scores(players, punt_categories=["to"])

comparison = balanced[["player", "total_9cat_z"]].rename(
    columns={"total_9cat_z": "balanced"}
)

comparison = comparison.merge(
    punt_ft[["player", "total_9cat_z"]].rename(columns={"total_9cat_z": "punt_ft"}),
    on="player",
)

comparison = comparison.merge(
    punt_to[["player", "total_9cat_z"]].rename(columns={"total_9cat_z": "punt_to"}),
    on="player",
)

comparison.round(2).sort_values("balanced", ascending=False)

,player,balanced,punt_ft,punt_to
0,Shai Gilgeous-Alexander,4.28,3.07,4.62
1,Nikola Jokic,3.64,3.76,5.30
2,Anthony Davis,3.07,3.22,3.24
3,Stephen Curry,-1.20,-2.31,0.14
4,DeMar DeRozan,-1.40,-1.89,-1.90
5,Myles Turner,-1.98,-1.44,-2.98
6,Nic Claxton,-2.57,-0.40,-3.74
7,Klay Thompson,-3.85,-4.01,-4.68


In [5]:
output_path = PROJECT_ROOT / "data" / "processed" / "notebook_sample_player_rankings.csv"

balanced.to_csv(output_path, index=False)

output_path

WindowsPath('C:/Users/graha/Documents/Data_Projects/nba-fantasy-decision-system/data/processed/notebook_sample_player_rankings.csv')